# Tecan Carousel: a plate hotel on the transfer station

```{device-card} tecan-carousel
```

| Property | Value |
|---|---|
| Transport | Operator-selected serial port, 9600 baud, 8E1, RTS/CTS |
| Protocol | StoreX remote operation: `CR` opens the link, one `<payload>\r\n` reply per command |
| Scope | Status reads, positioning, plate handling between cartridge positions and the transfer station |
| Safety | Only `setup()` is passive; positioning and every handling operation move the machine |

The unit is sold and documented by Tecan and manufactured by Liconic, whose StoreX command set it
speaks. It is named after the brand on the machine, not the manufacturer, and registers as
`tecan-carousel` with an OEM link to Liconic.

**What has been run on hardware.** The `CR`/`CC` handshake, every status read, `ST 1801`
initialization, rotation through `WR DM0` on all ten cartridge addresses, `ST 1905` export,
`ST 1904` import, `ST 1808`-free internal moves (`ST 1908` pick and `ST 1909` place), `ST 1800` soft
reset and `ST 1906` placement have all been exercised on one LPT220 EVO unit. The barcode
positioning mode (`ST 1910`) is deliberately not exposed.

**Positions are counted from the top.** Slot 1 is the topmost plate position in every cartridge.
The controller counts levels along the lift stroke; how many levels one physical slot occupies
depends on the cassette's shelf pitch. On this unit a 22-slot standard cassette maps one level per
slot, and a 7-slot deep-well cassette maps three levels per slot. The deep-well cassette counts its
controller levels upward from the bottom, so its top slot 1 sits at level 19 and its bottom slot 7 at
level 1. Set `cassette_levels_increase_downward=False` for such a cartridge. `level_for_slot()` does
that arithmetic for you; the mapping has been measured here for a deep-well cassette.

In [ ]:
# **HARDWARE: do not execute without per-run operator approval.**
from pylabrobot.tecan import TecanCarousel

## How it talks

`setup()` opens PLR's serial transport, clears the input buffer, sends `CR\r` and requires
`CC\r\n`. Every later command is answered by exactly one CRLF-terminated frame. Handling commands
(`ST …`) are acknowledged with `OK` and complete when the ready bit returns to `1`; the driver polls
`RD 1915` and `RD 1814` instead of sleeping a fixed time, and polls the plate sensors after a
handling command instead of trusting a single instant read.

Nothing is retried. A timeout, a malformed frame, an I/O failure or an in-flight cancellation faults
the session, which then requires `stop()` and `setup()`. A controller rejection (`E0`-`E5`) is
reported as `CarouselRejected` but does not fault the session, because the link is still in sync.
Closing the serial port is not an emergency stop.

Handling also requires an *active* handler: after a power cycle `RD 1801` reads `0` and rotation
commands are acknowledged but do nothing. `initialize()` sends `ST 1801` and homes the machine.


## Physical setup

Have an operator confirm the serial cable and port, the installed cassettes, and the clear motion
path. There is deliberately no default port.

The cartridge geometry is an installation detail: which positions hold which cassette type, how
many controller levels one physical slot occupies for each, and which way the controller's level
numbering runs. Pass it in so the API can talk in slots counted from the top. The example below
describes this lab's unit: positions 1-8 deep well with 7 slots and three levels per slot counting
upward from the bottom, positions 9-10 standard with 22 slots and one level per slot counting
downward from the top.

In [ ]:
# **HARDWARE: do not execute without per-run operator approval.**
port = "REPLACE_WITH_OPERATOR_SELECTED_PORT"
carousel = TecanCarousel(
  port=port,
  number_of_cartridges=10,
  levels_per_cartridge=22,
  cassette_slots={**{n: 7 for n in range(1, 9)}, 9: 22, 10: 22},
  cassette_level_step={**{n: 3 for n in range(1, 9)}, 9: 1, 10: 1},
  # Deep-well cassettes count controller levels upward from the bottom; standard
  # cassettes count downward from the top. Public positions are top-first either way.
  cassette_levels_increase_downward={**{n: False for n in range(1, 9)}, 9: True, 10: True},
  # Measured on cartridge 1: its 7-slot deep-well cassette sits at controller levels 22..4,
  # so slot 1 is level 22, slot 4 is level 13 and slot 7 is level 4. The other deep-well
  # cartridges were not measured.
  cassette_top_level={1: 22},
)

## Connect, which does not initialize

`setup()` opens the link and completes the handshake only. It does not activate, home or move the
machine, and a ready controller is not proof of a safe mechanism.


In [ ]:
# **HARDWARE: do not execute without per-run operator approval.**
await carousel.setup()

## Read the whole state

One serialized snapshot of the ready, error, active and plate-ready flags, both plate sensors, the
selected cartridge and level, and the raw `DM200` process word.

`process_status` is deliberately not decoded: this unit reports values the documented error table
does not describe, so the driver passes the raw word through rather than inventing a meaning.


In [ ]:
# **HARDWARE: do not execute without per-run operator approval.**
status = await carousel.get_status()
status

## Initialize

`initialize()` activates and homes the handling with `ST 1801`. It reads the activation flag first
and does nothing when the handling is already active, so it is safe to call after every connection.

**MOTION:** the carousel rotates and the lift homes.


In [ ]:
# **HARDWARE: do not execute without per-run operator approval.** MOTION.
await carousel.initialize()

## Rotate to a cartridge

`select_cartridge()` sends `WR DM0` and verifies both the selected address and the position the
controller reports. It does not touch `DM5`, so the lift stays where it is, and it does not approach
the transfer station, so a plate waiting there does not block it.

**MOTION:** the rotating plate turns.


In [ ]:
# **HARDWARE: do not execute without per-run operator approval.** MOTION.
await carousel.select_cartridge(1)

## Slots and levels

Ask the driver to translate, and pass controller levels to the operations. Slots are counted from
the top. For the deep-well cassette in position 1 that gives top slot 1 at level 19 and bottom slot
7 at level 1. For the standard cassette in position 9 it gives top slot 1 at level 1 and bottom
slot 22 at level 22.

In [ ]:
# **HARDWARE: do not execute without per-run operator approval.**
print("cartridge 1, slots 1 and 7 ->", carousel.level_for_slot(1, 1), carousel.level_for_slot(1, 7))
print("cartridge 9, slots 1 and 22 ->", carousel.level_for_slot(9, 1), carousel.level_for_slot(9, 22))
print("level 19 on cartridge 1 is slot", carousel.slot_for_level(1, 19))

## Retrieve a plate to the transfer station

`export_plate(cartridge, level)` rotates and lifts, picks the plate and places it on the transfer
station (`ST 1905`). It refuses to start unless the handler is ready, the shovel is empty and the
station is empty, and it raises if the station does not sense the plate afterwards.

The example takes the top slot of cartridge 1.

In [ ]:
# **HARDWARE: do not execute without per-run operator approval.** MOTION.
await carousel.export_plate(1, carousel.level_for_slot(1, 1))

## Return the plate into the cassette

`import_plate(cartridge, level)` takes the plate from the transfer station into a slot (`ST 1904`).
It requires a plate on the station and an empty shovel.


In [ ]:
# **HARDWARE: do not execute without per-run operator approval.** MOTION.
await carousel.import_plate(1, carousel.level_for_slot(1, 1))

## Move a plate inside a cartridge

`move_plate()` picks at the source level and places at the destination level (`ST 1908`, then
`ST 1909`), verifying the pick before it travels, so a failed pick never becomes a placement into an
unknown position. The transfer station is not used and must be empty.

This moves the plate from the top slot to the bottom slot of the same deep-well cassette.

In [ ]:
# **HARDWARE: do not execute without per-run operator approval.** MOTION.
await carousel.move_plate(
  1, carousel.level_for_slot(1, 1), 1, carousel.level_for_slot(1, 7)
)

## Recover a plate left on the shovel

`put_plate_on_transfer_station()` sends `ST 1906`, which places the plate already on the shovel onto
the transfer station without starting a new cartridge access. `soft_reset()` (`ST 1800`) releases an
interrupted access without moving the plate.

Both are recovery steps: after them the plate's physical position still needs the operator's eyes.


In [ ]:
# **HARDWARE: do not execute without per-run operator approval.** MOTION.
await carousel.soft_reset()
await carousel.put_plate_on_transfer_station(1, carousel.level_for_slot(1, 1))

## Clear a latched handling error

`reset()` sends `ST 1900`: it clears the error flag and `DM200`, re-homes the mechanism and leaves
the handling inactive, so `initialize()` follows. It is the documented way to clear a latched error,
not a retry of whatever failed.


In [ ]:
# **HARDWARE: do not execute without per-run operator approval.** MOTION.
await carousel.reset()
await carousel.initialize()

## Close the connection

`stop()` sends `CQ` to end the controller conversation and closes the port. It does not stop motion,
deactivate the controller or make manual loading safe. Always close in a `finally` block in
application code.


In [ ]:
# **HARDWARE: do not execute without per-run operator approval.**
await carousel.stop()